# 🎬 Movie Reviews — Text & Sentiment Analysis

**Author:** Rohit Malviya | Roll No: 4816  
**GitHub:** [@lokeshchawan](https://github.com/lokeshchawan)  
**Guide:** Prof. Dipti Shardul Khiste  
**Institution:** Pillai College of Arts, Commerce & Science (Autonomous), New Panvel  
**Program:** Master of Data Analytics — Semester 3 (2025–26)

---

This notebook performs:
- **Topic Modeling** using Latent Dirichlet Allocation (LDA)
- **Sentiment Analysis** using Logistic Regression with TF-IDF
- **Visualization** of topic distributions


## 📦 1. Install & Import Libraries

In [ ]:
# Install required libraries (for Google Colab)
# !pip install nltk scikit-learn matplotlib seaborn pandas numpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')
nltk.download('punkt')

print('✅ All libraries imported successfully!')

## 📂 2. Load Dataset

> This notebook uses the **IMDB Movie Reviews** dataset (50,000 reviews).  
> You can download it from [Kaggle](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) or load it directly below.

In [ ]:
# Option 1: Load from Kaggle (if kaggle API is configured)
# !kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
# !unzip imdb-dataset-of-50k-movie-reviews.zip

# Option 2: Load directly
# If you have the CSV file uploaded:
# df = pd.read_csv('IMDB Dataset.csv')

# For demonstration — generating a small sample dataset
sample_reviews = [
    ("This movie was absolutely fantastic! The acting was superb.", "positive"),
    ("Terrible film. Worst I have ever seen. Complete waste of time.", "negative"),
    ("A masterpiece of science fiction. Mind-blowing visuals.", "positive"),
    ("The plot was confusing and the direction was poor.", "negative"),
    ("Hilarious comedy! I laughed throughout the entire movie.", "positive"),
    ("The horror scenes were genuinely terrifying. Great thriller.", "positive"),
    ("Boring storyline with no character development at all.", "negative"),
    ("One of the best dramas I have ever watched. Deeply emotional.", "positive"),
    ("The war sequences were breathtaking and realistic.", "positive"),
    ("Awful acting and a predictable plot. Very disappointing.", "negative"),
    ("A haunting psychological thriller that stays with you.", "positive"),
    ("The comedy felt forced and the jokes fell flat.", "negative"),
    ("Brilliant sci-fi with amazing special effects.", "positive"),
    ("The movie scared me to death, I hate ghosts, horrifying.", "negative"),
    ("Wonderful performances and a touching story.", "positive"),
]

df = pd.DataFrame(sample_reviews, columns=['review', 'sentiment'])
print(f'Dataset Shape: {df.shape}')
print(f'\nSentiment Distribution:')
print(df['sentiment'].value_counts())
df.head()

## 🧹 3. Data Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Clean and preprocess raw review text."""
    # Lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['cleaned_review'] = df['review'].apply(preprocess_text)

print('✅ Preprocessing complete!')
print('\nSample cleaned review:')
print(df['cleaned_review'][0])

## 📐 4. TF-IDF Vectorization

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000, max_df=0.95, min_df=1)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['cleaned_review'])

print(f'TF-IDF Matrix Shape: {tfidf_matrix.shape}')
print('✅ TF-IDF vectorization complete!')

## 🗂️ 5. Topic Modeling with LDA

In [ ]:
# Fit LDA model
lda_model = LatentDirichletAllocation(n_components=7, random_state=42)
lda_topics = lda_model.fit_transform(tfidf_matrix)

# Define topic labels
topic_labels = {
    0: 'Sci-Fi & War Films',
    1: 'Criticism & Audience Reactions',
    2: 'General Movie Opinions',
    3: 'Filmmaking & Acting Quality',
    4: 'Emotional & Character-Driven Stories',
    5: 'Horror & Thriller Movies',
    6: 'Comedy & Popular Films'
}

# Assign dominant topic to each review
df['topic'] = lda_topics.argmax(axis=1)
df['topic_label'] = df['topic'].map(topic_labels)

print('✅ LDA Topic Modeling complete!')
print('\nTop words per topic:')
feature_names = tfidf_vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-6:-1]]
    print(f"Topic {topic_idx} [{topic_labels[topic_idx]}]: {', '.join(top_words)}")

## 🔍 6. Predict Topic for a New Review

In [ ]:
def predict_topic(review_text):
    """Predict the dominant topic for a given review."""
    cleaned = preprocess_text(review_text)
    vectorized = tfidf_vectorizer.transform([cleaned])
    topic_dist = lda_model.transform(vectorized)
    dominant_topic = topic_dist.argmax()
    return topic_labels[dominant_topic]

# Example
new_review1 = "The special effects were out of this world, a true sci-fi masterpiece."
new_review2 = "this movie scared me to death, i hate ghosts, they're horrifying"

print(f'Review: "{new_review1}"')
print(f'Predicted Topic: {predict_topic(new_review1)}')
print()
print(f'Review: "{new_review2}"')
print(f'Predicted Topic: {predict_topic(new_review2)}')

## 😊😠 7. Sentiment Analysis with Logistic Regression

In [ ]:
# Encode labels
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Features and labels
X = tfidf_matrix
y = df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train Logistic Regression
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = model.score(X_test, y_test)
print(f'Model Accuracy: {accuracy:.2f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

## 🔍 8. Predict Sentiment for New Reviews

In [ ]:
def predict_sentiment(review_text):
    """Predict the sentiment of a given review."""
    cleaned = preprocess_text(review_text)
    vectorized = tfidf_vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    return 'Positive 😊' if prediction == 1 else 'Negative 😠'

# Example usage
print(predict_sentiment("This movie was fantastic! I loved it."))
print(predict_sentiment("The worst film I have ever seen."))

## 📊 9. Visualization — Topic Distribution

In [ ]:
topic_counts = df['topic_label'].value_counts().reindex(topic_labels.values())

plt.figure(figsize=(14, 6))
bars = plt.bar(topic_counts.index, topic_counts.values, color='steelblue', edgecolor='white', width=0.6)
plt.title('Distribution of Topics in Movie Reviews', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Topic', fontsize=12)
plt.ylabel('Number of Reviews', fontsize=12)
plt.xticks(rotation=30, ha='right', fontsize=10)
plt.tight_layout()
plt.savefig('topic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualization saved as topic_distribution.png')

## 🔥 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix — Sentiment Classification', fontsize=13, fontweight='bold')
plt.xlabel('Predicted', fontsize=11)
plt.ylabel('Actual', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved as confusion_matrix.png')

## ✅ Conclusion

This project successfully applied NLP techniques to analyze movie reviews:

- **LDA** effectively grouped reviews into **7 meaningful topic clusters**, ranging from Sci-Fi & War Films to Comedy & Popular Films.
- **Logistic Regression** with TF-IDF achieved **89% accuracy** in classifying reviews as Positive or Negative.
- These insights can power better **recommendation systems** and **audience sentiment dashboards**.

### 🔮 Future Enhancements
- Deep learning models (LSTM, BERT) for improved sentiment accuracy
- Hierarchical topic modeling for finer-grained analysis
- Real-time sentiment prediction web app
- Multilingual review support

---
*Submitted by Rohit Malviya (Roll No: 4816) | GitHub: @lokeshchawan | Pillai College, New Panvel — MDA Sem 3, 2025–26*